# Prompting Lightweight Granite 4.0 H 350M for JSON Classification

This Colab notebook teaches a practical, reliable prompting pattern for basic text classification with a small local model:

`unsloth/granite-4.0-h-350m-GGUF`

The focus is small, repeatable classification tasks that return structured JSON.

## Revision note

This version fixes the common error:

```text
ValueError: No JSON object found
```

That error happens when the model returns an empty string, a bare label such as `positive`, or extra text instead of a JSON object. This notebook now uses:

1. Stronger few-shot prompt templates that end with `JSON:`.
2. Raw-output debugging so you can see exactly what the model produced.
3. A brace-aware JSON extractor instead of a fragile regular expression.
4. Schema validation before trusting outputs.
5. Safe coercion for simple label-only outputs.
6. Retry and repair logic before raising an error.
7. A last-resort tutorial fallback for sentiment examples so early cells do not stop the lesson.

## What you will learn

1. Why lightweight 350M parameter models need narrow prompts.
2. How to design compact label sets.
3. How to force predictable JSON shape.
4. How to validate outputs with Python.
5. How to retry safely when JSON is malformed.
6. How to run practice classification exercises in batches.

## Practical rule

Small models are useful when the task is narrow, the labels are explicit, the output schema is small, and the prompt avoids unnecessary reasoning.

## 1. Runtime setup

In Colab, choose:

`Runtime` → `Change runtime type` → `CPU`

A GPU can be faster, but this notebook defaults to CPU for portability. The model is small enough for basic practice on standard Colab.

In [ ]:
# Install minimal dependencies.
# The extra index asks pip to prefer a prebuilt CPU wheel for llama-cpp-python when available.

!pip -q install --upgrade huggingface_hub jsonschema pandas tqdm
!pip -q install --upgrade llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

## 2. Select and download the GGUF model

We will use a GGUF quantized model from Hugging Face.

Default choice:

`granite-4.0-h-350m-UD-Q4_K_XL.gguf`

Why this choice?

- It is small enough for Colab CPU practice.
- It gives better quality than the most aggressive 1-bit or 2-bit quantizations.
- It is still much smaller than typical multi-billion parameter models.

If download fails because the repo changed, run the file listing cell and pick a different `.gguf` filename.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
from pathlib import Path

MODEL_REPO = "unsloth/granite-4.0-h-350m-GGUF"
MODEL_FILE = "granite-4.0-h-350m-UD-Q4_K_XL.gguf"

print("Available GGUF files in the repo:")
for f in list_repo_files(MODEL_REPO):
    if f.endswith(".gguf"):
        print("-", f)

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
)

print("\nDownloaded model path:", model_path)
print("Size on disk MB:", round(Path(model_path).stat().st_size / 1_000_000, 1))

## 3. Load the model with conservative inference settings

For classification, prefer deterministic settings:

- `temperature=0.0`
- small `max_tokens`
- short prompts
- explicit labels
- explicit JSON schema

For a 350M model, do not ask for long explanations. Ask for one compact object.

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=1024,
    n_threads=2,
    n_gpu_layers=0,
    verbose=False,
)

print("Model loaded.")

## 4. Core utilities: generation, extraction, validation, and retry

The earlier notebook failed because it tried to extract `{...}` from the raw model output, but the model did not produce a JSON object.

The fix is to treat model output as untrusted text:

1. Generate text.
2. Print raw text when debugging.
3. Extract JSON only if JSON exists.
4. Validate the parsed object against a schema.
5. Coerce simple bare-label outputs when safe.
6. Retry with a repair prompt when needed.

The helper functions below are reusable across all classification tasks.

In [ ]:
import json
import re
from typing import Any, Optional
from jsonschema import validate, ValidationError


def generate_text(prompt: str, max_tokens: int = 80, debug: bool = False) -> str:
    """Run a deterministic completion and return raw model text."""
    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=1.0,
        repeat_penalty=1.05,
        stop=["<|endoftext|>", "<|end_of_text|>", "\n\n\n"],
    )
    raw = response["choices"][0].get("text", "")
    if debug:
        print("RAW MODEL OUTPUT:")
        print(repr(raw))
    return raw.strip()


def extract_first_json_object(text: str) -> str:
    """Extract the first complete JSON object using brace counting."""
    if not text or not text.strip():
        raise ValueError("Empty model output")

    start = text.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in output: {text!r}")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]

    raise ValueError(f"Started JSON object but did not find closing brace: {text!r}")


def enum_values_for_field(schema: dict, field: str) -> list[str]:
    return schema["properties"][field]["enum"]


def coerce_bare_enum_output(raw_text: str, schema: dict, field: str) -> Optional[dict]:
    """Convert a bare model output like 'positive' into {'label':'positive'} when it is safe."""
    allowed = enum_values_for_field(schema, field)
    cleaned = raw_text.strip().lower()
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()
    cleaned = cleaned.strip(" .,:;!?'\"`\n\t")

    if cleaned in allowed:
        return {field: cleaned}

    # Also handle outputs such as 'label: positive' or 'The answer is positive'.
    hits = []
    for value in allowed:
        if re.search(rf"\b{re.escape(value)}\b", cleaned):
            hits.append(value)

    if len(hits) == 1:
        return {field: hits[0]}

    return None


def parse_and_validate(
    raw_text: str,
    schema: dict,
    single_enum_field: Optional[str] = None,
    allow_bare_enum: bool = True,
) -> dict:
    """Parse model output into JSON and validate it. Optionally coerce simple bare labels."""
    try:
        json_text = extract_first_json_object(raw_text)
        data = json.loads(json_text)
        validate(instance=data, schema=schema)
        return data
    except Exception as json_error:
        if allow_bare_enum and single_enum_field:
            coerced = coerce_bare_enum_output(raw_text, schema, single_enum_field)
            if coerced is not None:
                validate(instance=coerced, schema=schema)
                return coerced
        raise json_error


def schema_hint(schema: dict) -> str:
    """Create a compact schema hint for repair prompts."""
    return json.dumps(schema.get("properties", {}), ensure_ascii=False)


def classify_with_repair(
    text: str,
    prompt_builder,
    schema: dict,
    max_tokens: int = 80,
    single_enum_field: Optional[str] = None,
    retries: int = 2,
    debug: bool = False,
) -> dict:
    """Classify text, validate output, and retry with a repair prompt if needed."""
    raw_outputs = []
    prompt = prompt_builder(text)

    for attempt in range(retries + 1):
        raw = generate_text(prompt, max_tokens=max_tokens, debug=debug)
        raw_outputs.append(raw)

        try:
            return parse_and_validate(
                raw,
                schema,
                single_enum_field=single_enum_field,
                allow_bare_enum=True,
            )
        except Exception as error:
            if debug:
                print(f"Attempt {attempt + 1} failed: {type(error).__name__}: {error}")

            prompt = f"""
You are repairing a classification output.
Return one valid JSON object only.
No markdown. No explanation.

Input text:
{text!r}

Previous invalid model output:
{raw!r}

Required JSON properties:
{schema_hint(schema)}

Return the corrected JSON now.
JSON:
""".strip()

    raise ValueError(
        "Could not produce valid JSON after retries. "
        f"Raw outputs were: {raw_outputs!r}"
    )

## 5. Sentiment classification with a stronger prompt

This prompt uses examples and ends with `JSON:`. That final cue matters for small completion models because it gives the model a concrete place to continue.

This cell uses `classify_sentiment_safe`, which retries and then falls back to a small keyword heuristic only for tutorial continuity. For real evaluation, use the model output and validation report rather than hiding failures.

In [ ]:
SENTIMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ["positive", "negative", "neutral"]}
    },
    "required": ["label"],
    "additionalProperties": False,
}


def build_sentiment_prompt(text: str) -> str:
    return f"""
You are a strict sentiment classification function.
Return only valid JSON. No markdown. No explanation.

Allowed labels:
- positive: praise, satisfaction, gratitude, approval, or success
- negative: complaint, frustration, rejection, failure, or dissatisfaction
- neutral: factual, unclear, mixed, or no strong sentiment

Schema:
{{"label":"positive|negative|neutral"}}

Examples:
Input: "The setup was simple and worked perfectly."
JSON: {{"label":"positive"}}

Input: "The login flow failed twice and wasted my time."
JSON: {{"label":"negative"}}

Input: "The ticket was opened on Tuesday."
JSON: {{"label":"neutral"}}

Input: {text!r}
JSON:
""".strip()


def heuristic_sentiment_fallback(text: str) -> dict:
    """Last-resort fallback so beginner tutorial cells keep running."""
    lower = text.lower()
    positive_words = ["good", "great", "simple", "worked", "perfect", "perfectly", "thanks", "thank", "love", "helpful", "easy"]
    negative_words = ["bad", "broken", "failed", "fail", "error", "wasted", "angry", "refund", "ignored", "crash", "problem"]

    pos = sum(word in lower for word in positive_words)
    neg = sum(word in lower for word in negative_words)

    if pos > neg:
        return {"label": "positive"}
    if neg > pos:
        return {"label": "negative"}
    return {"label": "neutral"}


def classify_sentiment(text: str, debug: bool = False) -> dict:
    return classify_with_repair(
        text=text,
        prompt_builder=build_sentiment_prompt,
        schema=SENTIMENT_SCHEMA,
        max_tokens=60,
        single_enum_field="label",
        retries=2,
        debug=debug,
    )


def classify_sentiment_safe(text: str, debug: bool = False) -> dict:
    try:
        return classify_sentiment(text, debug=debug)
    except Exception as error:
        if debug:
            print("Model classification failed after retries. Using tutorial fallback.")
            print(type(error).__name__, error)
        return heuristic_sentiment_fallback(text)

# This line should no longer crash if the model returns an empty string or a bare label.
print(classify_sentiment_safe("The setup was simple and the output worked perfectly.", debug=True))

## 6. Practice cell: sentiment classification

Edit the examples and rerun the cell.

Notice that each item gets only one label. Avoid labels that overlap too much. Small models perform better when labels are mutually exclusive.

In [ ]:
examples = [
    "The product arrived early and works exactly as expected.",
    "I have not received my refund and support has ignored me twice.",
    "The ticket was created on Tuesday and assigned to the billing queue.",
    "The app is useful, but the latest update made the login screen slower.",
]

for text in examples:
    print(text)
    print(classify_sentiment_safe(text, debug=False))
    print()

## 7. Why the first version failed

The earlier parser did this:

```python
match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
if not match:
    raise ValueError("No JSON object found")
```

That is fine only when the model definitely returns JSON. Lightweight completion models often return:

```text
positive
```

or:

```text
The label is positive.
```

or an empty string if the prompt does not provide a strong continuation cue.

The revised notebook avoids that failure by:

1. Ending prompts with `JSON:`.
2. Accepting safe bare-label outputs for single-label tasks.
3. Retrying with a repair prompt.
4. Printing raw output in debug mode.

## 8. Enterprise ticket routing

Classification is most useful when labels map to actions.

For ticket routing, keep the labels small and concrete:

- `billing`
- `technical_support`
- `account_access`
- `sales`
- `other`

The output includes two fields:

- `category`: one routing label
- `urgent`: boolean

This is still simple enough for a 350M model, but it is harder than sentiment because the model must produce two fields.

In [ ]:
TICKET_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["billing", "technical_support", "account_access", "sales", "other"],
        },
        "urgent": {"type": "boolean"},
    },
    "required": ["category", "urgent"],
    "additionalProperties": False,
}


def build_ticket_prompt(text: str) -> str:
    return f"""
You are a strict support ticket routing function.
Return only valid JSON. No markdown. No explanation.

Choose exactly one category:
- billing: invoices, payment, refund, charge, subscription price
- technical_support: bug, error, crash, broken feature, setup problem
- account_access: login, password, locked account, MFA, permissions
- sales: pricing question before buying, demo request, plan comparison
- other: anything else

Urgent is true only when the user says production is blocked, service is down, security is at risk, or a deadline is immediate.

Schema:
{{"category":"billing|technical_support|account_access|sales|other","urgent":true|false}}

Examples:
Ticket: "I was charged twice."
JSON: {{"category":"billing","urgent":false}}

Ticket: "The API is down in production."
JSON: {{"category":"technical_support","urgent":true}}

Ticket: "I cannot log in after enabling MFA."
JSON: {{"category":"account_access","urgent":false}}

Ticket: "Can I schedule a demo for my team?"
JSON: {{"category":"sales","urgent":false}}

Ticket: "Please rename my workspace."
JSON: {{"category":"other","urgent":false}}

Ticket: {text!r}
JSON:
""".strip()


def classify_ticket(text: str, debug: bool = False) -> dict:
    return classify_with_repair(
        text=text,
        prompt_builder=build_ticket_prompt,
        schema=TICKET_SCHEMA,
        max_tokens=80,
        single_enum_field=None,
        retries=2,
        debug=debug,
    )

practice_tickets = [
    "I was charged twice this month and need a refund.",
    "We cannot log in after enabling MFA. Production deploy is blocked.",
    "Can someone explain the difference between the starter and pro plans?",
    "The dashboard crashes whenever I upload a CSV file.",
    "Please update the company name on our account profile.",
]

for ticket in practice_tickets:
    print(ticket)
    try:
        print(classify_ticket(ticket))
    except Exception as error:
        print("Could not classify as valid JSON:", type(error).__name__, error)
    print()

## 9. Practice cell: add your own ticket examples

Add five examples that are hard but realistic.

Good practice examples include:

- mixed billing and technical issues
- vague customer language
- urgent language that is emotional but not operationally urgent
- short one-sentence tickets
- long tickets with irrelevant details

In [ ]:
my_tickets = [
    "I need help changing my plan before renewal tomorrow.",
    "The API returns a 500 error for every request in our production app.",
    "I forgot my password and no longer have access to my authenticator app.",
    "Your product looks interesting. Do you offer a demo for enterprise teams?",
    "This is annoying and I want someone to call me back.",
]

for ticket in my_tickets:
    print(ticket)
    try:
        print(classify_ticket(ticket))
    except Exception as error:
        print("Could not classify as valid JSON:", type(error).__name__, error)
    print()

## 10. Batch evaluation with expected labels

Prompting is engineering. Measure it.

This cell compares model outputs to expected labels and reports simple accuracy. Rows that fail JSON validation are marked as invalid instead of stopping the notebook.

In [ ]:
import pandas as pd

labeled_tickets = [
    {
        "text": "My invoice is wrong and the tax amount looks too high.",
        "expected_category": "billing",
        "expected_urgent": False,
    },
    {
        "text": "The app is down for all users and our support queue is exploding.",
        "expected_category": "technical_support",
        "expected_urgent": True,
    },
    {
        "text": "I cannot reset my password because the email never arrives.",
        "expected_category": "account_access",
        "expected_urgent": False,
    },
    {
        "text": "Do you have annual pricing for a 50-seat team?",
        "expected_category": "sales",
        "expected_urgent": False,
    },
    {
        "text": "Please delete the duplicate workspace named test-company-old.",
        "expected_category": "other",
        "expected_urgent": False,
    },
]

rows = []
for item in labeled_tickets:
    try:
        prediction = classify_ticket(item["text"])
        rows.append({
            **item,
            "valid_json": True,
            "predicted_category": prediction["category"],
            "predicted_urgent": prediction["urgent"],
            "category_correct": prediction["category"] == item["expected_category"],
            "urgent_correct": prediction["urgent"] == item["expected_urgent"],
            "error": "",
        })
    except Exception as error:
        rows.append({
            **item,
            "valid_json": False,
            "predicted_category": None,
            "predicted_urgent": None,
            "category_correct": False,
            "urgent_correct": False,
            "error": f"{type(error).__name__}: {error}",
        })

df = pd.DataFrame(rows)
display(df)

print("Valid JSON rate:", df["valid_json"].mean())
print("Category accuracy:", df["category_correct"].mean())
print("Urgency accuracy:", df["urgent_correct"].mean())

## 11. Inspecting failures

When a model output fails validation, run the same item with `debug=True`.

The most useful thing to inspect is the raw model output. That tells you whether the problem is:

1. Empty output.
2. Bare label instead of JSON.
3. Extra explanation text.
4. Wrong field name.
5. Wrong label.
6. Malformed JSON.

In [ ]:
# Change this text to debug a failure.
debug_text = "The setup was simple and the output worked perfectly."

print(classify_sentiment_safe(debug_text, debug=True))

## 12. Prompt revision exercise

When the model fails, do not immediately switch models. First improve the task design.

Try these changes:

1. Reduce the number of labels.
2. Make label definitions more different from each other.
3. Add one example per label.
4. Remove optional fields from the JSON.
5. Lower `max_tokens` if the model rambles.
6. Add validation and retry.

For small models, classification prompts usually improve when they are shorter, more explicit, and less conversational.

## 13. Output-only JSON prompt template

Use this template when you create new tasks.

```text
You are a strict classification function.
Return only valid JSON. No markdown. No explanation.

Task:
<Classify what?>

Allowed labels:
- label_a: <clear definition>
- label_b: <clear definition>
- label_c: <clear definition>

Schema:
{"label":"label_a|label_b|label_c"}

Examples:
Input: "<example A>"
JSON: {"label":"label_a"}

Input: "<example B>"
JSON: {"label":"label_b"}

Input: "<new text>"
JSON:
```

Keep schemas small. For 350M models, one to three output fields is usually much safer than a deeply nested object.

## 14. Independent practice task: email triage

This task has one field and three labels. It is a good fit for a small model.

In [ ]:
EMAIL_SCHEMA = {
    "type": "object",
    "properties": {
        "action": {"type": "string", "enum": ["reply_needed", "archive", "escalate"]}
    },
    "required": ["action"],
    "additionalProperties": False,
}


def build_email_prompt(text: str) -> str:
    return f"""
You are a strict email triage classifier.
Return only valid JSON. No markdown. No explanation.

Allowed actions:
- reply_needed: sender asks a question or expects a response
- archive: informational message, receipt, newsletter, or no action needed
- escalate: legal, security, angry customer, executive request, or urgent business risk

Schema:
{{"action":"reply_needed|archive|escalate"}}

Examples:
Email: "Can you send the updated invoice before Friday?"
JSON: {{"action":"reply_needed"}}

Email: "Your package was delivered at 2:14 PM."
JSON: {{"action":"archive"}}

Email: "The customer is threatening to cancel unless this outage is resolved today."
JSON: {{"action":"escalate"}}

Email: {text!r}
JSON:
""".strip()


def classify_email(text: str, debug: bool = False) -> dict:
    return classify_with_repair(
        text=text,
        prompt_builder=build_email_prompt,
        schema=EMAIL_SCHEMA,
        max_tokens=60,
        single_enum_field="action",
        retries=2,
        debug=debug,
    )

emails = [
    "Can you send the updated invoice before Friday?",
    "Your package was delivered at 2:14 PM.",
    "The customer is threatening to cancel unless this outage is resolved today.",
]

for email in emails:
    print(email)
    try:
        print(classify_email(email))
    except Exception as error:
        print("Could not classify as valid JSON:", type(error).__name__, error)
    print()

In [ ]:
# Practice: write your own classifier here.
# 1. Define a JSON schema.
# 2. Write a prompt builder that ends with JSON:.
# 3. Use classify_with_repair.
# 4. Test five examples.

# Your code here.

## 15. Checklist for lightweight model prompting

Before blaming the model, check your task design.

- Are there five labels or fewer?
- Are labels mutually exclusive?
- Is the schema one flat JSON object?
- Are you using `temperature=0.0`?
- Is `max_tokens` small?
- Did you define every label?
- Does the prompt end with `JSON:`?
- Did you include examples that show the exact output shape?
- Did you validate the output with Python?
- Did you inspect raw failures with `debug=True`?
- Did you test on labeled examples?

For a 350M parameter model, the best prompt is usually short, explicit, and boring.

## 16. Summary

The correct approach for small local classification models is:

1. Make the task narrow.
2. Use explicit labels.
3. Use a tiny JSON schema.
4. End the prompt with a strong output cue such as `JSON:`.
5. Use deterministic generation.
6. Validate outputs.
7. Coerce safe bare-label outputs only when the schema has one enum field.
8. Retry or repair malformed JSON.
9. Measure accuracy on examples.

This pattern is production-friendly because the model is only one part of the system. The validator, schema, and evaluation loop are just as important as the prompt.